In [0]:
table_name = "fact_claims"
silver_table_fqn = f"he_prod_incen_analytics_dbw_01.silver.{table_name}"

df = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.fact_claims")
display(df.limit(5))

In [0]:
from pyspark.sql.functions import col, upper, trim, when, to_date, row_number, abs as spark_abs
from pyspark.sql.window import Window

# 1. Read Bronze Dimensions to map Source SKs -> Business Keys
df_map_member = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_members").select(col("member_sk").cast("int").alias("src_member_sk"), col("member_id"))
df_map_provider = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_providers").select(col("provider_sk").cast("int").alias("src_provider_sk"), col("provider_id"))
df_map_facility = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_facilities").select(col("facility_sk").cast("int").alias("src_facility_sk"), col("facility_id"))
df_map_service = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_service_types").select(col("service_type_sk").cast("int").alias("src_service_type_sk"), col("service_type_id"))
df_map_icd10 = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_icd10_codes").select(col("icd10_code_sk").cast("int").alias("src_icd10_code_sk"), col("icd10_code"))
df_map_cpt = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_cpt_codes").select(col("cpt_code_sk").cast("int").alias("src_cpt_code_sk"), col("cpt_code"))
df_map_plan = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_plan_types").select(col("plan_type_sk").cast("int").alias("src_plan_type_sk"), col("plan_type_id"))
df_map_network = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_network_status").select(col("network_status_sk").cast("int").alias("src_network_status_sk"), col("network_status_id"))
df_map_claim_status = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_claim_status").select(col("claim_status_sk").cast("int").alias("src_claim_status_sk"), col("claim_status_id"))

# 2. Enrich Fact Table: Join Source SKs to get Business Keys
df = df.join(df_map_member, df["member_sk"] == df_map_member["src_member_sk"], "left").drop("src_member_sk", "member_sk")
df = df.join(df_map_provider, df["provider_sk"] == df_map_provider["src_provider_sk"], "left").drop("src_provider_sk", "provider_sk")
df = df.join(df_map_facility, df["facility_sk"] == df_map_facility["src_facility_sk"], "left").drop("src_facility_sk", "facility_sk")
df = df.join(df_map_service, df["service_type_sk"] == df_map_service["src_service_type_sk"], "left").drop("src_service_type_sk", "service_type_sk")
df = df.join(df_map_icd10, df["icd10_code_sk"] == df_map_icd10["src_icd10_code_sk"], "left").drop("src_icd10_code_sk", "icd10_code_sk")
df = df.join(df_map_cpt, df["cpt_code_sk"] == df_map_cpt["src_cpt_code_sk"], "left").drop("src_cpt_code_sk", "cpt_code_sk")
df = df.join(df_map_plan, df["plan_type_sk"] == df_map_plan["src_plan_type_sk"], "left").drop("src_plan_type_sk", "plan_type_sk")
df = df.join(df_map_network, df["network_status_sk"] == df_map_network["src_network_status_sk"], "left").drop("src_network_status_sk", "network_status_sk")
df = df.join(df_map_claim_status, df["claim_status_sk"] == df_map_claim_status["src_claim_status_sk"], "left").drop("src_claim_status_sk", "claim_status_sk")

# 3. Standardize Text on the newly fetched Business Keys
text_cols = ["claim_id", "member_id", "provider_id", "facility_id", "service_type_id", 
             "icd10_code", "cpt_code", "plan_type_id", "network_status_id", "claim_status_id",
             "denial_reason", "claim_type", "appeal_status"]
for c in text_cols:
    df = df.withColumn(c, upper(trim(col(c))))

# 4. Safe Cast Dates
for date_col in ["service_date", "claim_received_date", "claim_processed_date", "payment_date", "record_created_date", "record_modified_date"]:
    df = df.withColumn(date_col, to_date(trim(col(date_col))))

# 5. Safe Cast Integers
for int_col in ["days_to_process", "days_to_pay", "units_of_service", "quantity"]:
    df = df.withColumn(int_col, trim(col(int_col)).cast("int"))
    if int_col in ["days_to_process", "days_to_pay"]:
        df = df.withColumn(int_col, when(col(int_col) < 0, 0).otherwise(col(int_col)))

# 6. Precision Optimization (Financials)
for dec_col in ["total_charge", "contractual_adjustment", "allowed_amount", "deductible_applied", 
                "co_pay_amount", "copay_applied", "insurance_paid_amount", "member_responsibility", 
                "provider_write_off", "appeal_amount"]:
    is_valid = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid, trim(col(dec_col)).cast("decimal(15,2)")).otherwise(None))
    df = df.withColumn(dec_col, spark_abs(col(dec_col)))

# 7. Boolean Mapping
for bool_col in ["prior_auth_required", "prior_auth_obtained", "emergency_service", "preventive_service", "special_handling", "appeal_flag"]:
    df = df.withColumn(bool_col, 
                       when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE"), True)
                       .when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE"), False)
                       .otherwise(None).cast("boolean"))

# 8. Deduplication
dedup_window = Window.partitionBy("claim_id").orderBy(col("_ingested_at").desc())
df = df.withColumn("_dq_is_deduped", row_number().over(dedup_window))
df = df.filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped")

# 9. Drop audit/source columns
df = df.drop("_ingested_at", "_source_file", "_inserted_at", "claim_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(silver_table_fqn)

print(f"✅ Successfully wrote {table_name} to Silver layer.")